# 01 - Load and explore
Exercise 1: San Diego FC opposition report (SD vs Nashville, 25 Jul 2025)

1. StatsBomb schedule
2. SkillCorner schedule and linking
3. StatsBomb events
4. San Diego results
5. SkillCorner
6. Save

## Setup

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)

In [2]:
DATA_FOLDER = Path("../../data")
STATSBOMB_FOLDER = DATA_FOLDER / "StatsBomb Events"
SKILLCORNER_FOLDER = DATA_FOLDER / "SkillCorner Events"
PHYSICAL_FOLDER = DATA_FOLDER / "Physical Data"

## 1 - StatsBomb schedule

In [3]:
sb_schedule_raw = pd.read_csv(DATA_FOLDER / "StatsBomb_MatchSchedule.csv")
sb_schedule_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 62 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   match_id                   48 non-null     int64  
 1   match_date                 48 non-null     str    
 2   kick_off                   48 non-null     str    
 3   home_score                 48 non-null     float64
 4   away_score                 48 non-null     float64
 5   attendance                 48 non-null     int64  
 6   behind_closed_doors        48 non-null     bool   
 7   neutral_ground             48 non-null     bool   
 8   collection_status          48 non-null     str    
 9   play_status                48 non-null     str    
 10  match_status               48 non-null     str    
 11  match_status_360           48 non-null     str    
 12  last_updated               48 non-null     str    
 13  last_updated_360           48 non-null     str    
 14  match_w

Columns needed

In [4]:
columns_to_keep = [
    "match_id",
    "match_date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
]
sb_schedule = sb_schedule_raw[columns_to_keep].copy()

In [5]:
# turn the text "2025-02-24" into a real date, so we can compare dates
sb_schedule["match_date"] = pd.to_datetime(sb_schedule["match_date"])

# sort by date
sb_schedule = sb_schedule.sort_values("match_date")
sb_schedule = sb_schedule.reset_index(drop=True)
sb_schedule.head()

,match_id,match_date,home_team,away_team,home_score,away_score
0,3981145,2025-02-23,Nashville SC,New England Revolution,0.0,0.0
1,3981153,2025-02-24,LA Galaxy,San Diego FC,0.0,2.0
2,3981160,2025-03-02,San Diego FC,St. Louis City,0.0,0.0
3,3981156,2025-03-02,New York Red Bulls,Nashville SC,2.0,0.0
4,3981180,2025-03-09,Nashville SC,Portland Timbers,2.0,0.0


Date range check (all before 25 Jul 2025)

In [6]:
first_date = sb_schedule["match_date"].min()
last_date = sb_schedule["match_date"].max()
print("First match:", first_date.date())
print("Last match: ", last_date.date())

First match: 2025-02-23
Last match:  2025-07-20


### Focus team
Each match involves San Diego or Nashville. 

In [7]:
sd_is_home = sb_schedule["home_team"] == "San Diego FC"
sd_is_away = sb_schedule["away_team"] == "San Diego FC"
sd_plays = sd_is_home | sd_is_away            # | means OR

sb_schedule["focus_team"] = np.where(sd_plays, "San Diego", "Nashville")
sb_schedule["focus_team"].value_counts()

focus_team
Nashville    24
San Diego    24
Name: count, dtype: int64

## 2 - SkillCorner schedule
Different match IDs and team names from StatsBomb ("San Diego FC" vs "San Diego"), so link on date + focus team.

In [8]:
sc_schedule_raw = pd.read_csv(DATA_FOLDER / "SkillCorner_MatchSchedule.csv")
sc_schedule_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   id                      48 non-null     int64
 1   date_time               48 non-null     str  
 2   home_team               48 non-null     str  
 3   away_team               48 non-null     str  
 4   status                  48 non-null     str  
 5   last_modified           48 non-null     str  
 6   competition_id          48 non-null     int64
 7   season_id               48 non-null     int64
 8   competition_edition_id  48 non-null     int64
 9   date                    48 non-null     str  
 10  home_team_id            48 non-null     int64
 11  home_team_shortname     48 non-null     str  
 12  away_team_id            48 non-null     int64
 13  away_team_shortname     48 non-null     str  
 14  events_available        48 non-null     bool 
dtypes: bool(1), int64(6), str(8)
memory 

In [9]:
sc_schedule = sc_schedule_raw[["id", "date", "home_team_shortname", "away_team_shortname"]].copy()
sc_schedule = sc_schedule.rename(columns={"id": "sc_match_id", "date": "match_date"})
sc_schedule["match_date"] = pd.to_datetime(sc_schedule["match_date"])
sc_schedule.head()

,sc_match_id,match_date,home_team_shortname,away_team_shortname
0,2022782,2025-07-20,San Diego,Vancouver
1,2022787,2025-07-20,Nashville,Toronto
2,2022631,2025-07-17,San Diego,Toronto
3,2022635,2025-07-17,Nashville,Columbus Crew
4,2022426,2025-07-13,Chicago Fire,San Diego


In [10]:
sd_is_home = sc_schedule["home_team_shortname"] == "San Diego"
sd_is_away = sc_schedule["away_team_shortname"] == "San Diego"
sd_plays = sd_is_home | sd_is_away

sc_schedule["focus_team"] = np.where(sd_plays, "San Diego", "Nashville")
sc_schedule.head()

,sc_match_id,match_date,home_team_shortname,away_team_shortname,focus_team
0,2022782,2025-07-20,San Diego,Vancouver,San Diego
1,2022787,2025-07-20,Nashville,Toronto,Nashville
2,2022631,2025-07-17,San Diego,Toronto,San Diego
3,2022635,2025-07-17,Nashville,Columbus Crew,Nashville
4,2022426,2025-07-13,Chicago Fire,San Diego,San Diego


Merge the dataframes

In [11]:
sc_ids_only = sc_schedule[["match_date", "focus_team", "sc_match_id"]]

matches = sb_schedule.merge(sc_ids_only, on=["match_date", "focus_team"], how="left")

missing_links = matches["sc_match_id"].isna().sum()
print("Matches without a SkillCorner ID:", missing_links)
matches.head()

Matches without a SkillCorner ID: 0


,match_id,match_date,home_team,away_team,home_score,away_score,focus_team,sc_match_id
0,3981145,2025-02-23,Nashville SC,New England Revolution,0.0,0.0,Nashville,2004842
1,3981153,2025-02-24,LA Galaxy,San Diego FC,0.0,2.0,San Diego,2004990
2,3981160,2025-03-02,San Diego FC,St. Louis City,0.0,0.0,San Diego,2005659
3,3981156,2025-03-02,New York Red Bulls,Nashville SC,2.0,0.0,Nashville,2005657
4,3981180,2025-03-09,Nashville SC,Portland Timbers,2.0,0.0,Nashville,2006613


SkillCorner schedule lists 48 matches, but the SkillCorner Events folder has only 47 files. 

In [12]:
skillcorner_files = list(SKILLCORNER_FOLDER.glob("*.csv"))
print("Number of files:", len(skillcorner_files))

# '2010342_dynamic_events.csv' -> '2010342' -> 2010342
files_sc_ids = []
for file in skillcorner_files:
    id_text = file.name.split("_")[0]
    files_sc_ids.append(int(id_text))

has_file = matches["sc_match_id"].isin(files_sc_ids)
matches[~has_file]          # ~ means NOT

Number of files: 47


,match_id,match_date,home_team,away_team,home_score,away_score,focus_team,sc_match_id
19,3981284,2025-04-27,Nashville SC,Chicago Fire,7.0,2.0,Nashville,2013490


Nashville 7–2 Chicago is missing, which we dont need

## 3 - StatsBomb events
48 files, one per match, combined into one table.

In [13]:
statsbomb_files = sorted(STATSBOMB_FOLDER.glob("*_events.csv"))
print("Number of files:", len(statsbomb_files))

Number of files: 48


In [14]:
event_tables = []
for file in statsbomb_files:
    one_match = pd.read_csv(file, low_memory=False)
    event_tables.append(one_match)

events = pd.concat(event_tables, ignore_index=True)
events = events.copy()
events.shape

(178466, 150)

Columns check

In [15]:
events.columns.tolist()

['50_50',
 'attacking_direction',
 'ball_receipt_outcome',
 'ball_recovery_recovery_failure',
 'block_deflection',
 'carry_end_location',
 'clearance_aerial_won',
 'clearance_body_part',
 'clearance_duel_win_probability',
 'clearance_head',
 'clearance_left_foot',
 'clearance_right_foot',
 'control_degree',
 'counterpress',
 'defensive_responsibility',
 'dribble_outcome',
 'dribble_overrun',
 'duel_duel_win_probability',
 'duel_hops_opponent_rating',
 'duel_hops_opponent_raw_rating',
 'duel_hops_rating',
 'duel_hops_raw_rating',
 'duel_outcome',
 'duel_type',
 'duration',
 'foul_committed_advantage',
 'foul_committed_card',
 'foul_won_advantage',
 'foul_won_defensive',
 'goalkeeper_body_part',
 'goalkeeper_end_location',
 'goalkeeper_outcome',
 'goalkeeper_position',
 'goalkeeper_technique',
 'goalkeeper_type',
 'id',
 'index',
 'interception_outcome',
 'is_controlled_possession',
 'is_transition',
 'location',
 'location_bucket',
 'match_id',
 'minute',
 'miscontrol_aerial_won',
 'mis

In [16]:
# search for columns by keyword
pass_columns = [column for column in events.columns if "pass" in column]
pass_columns

['pass_aerial_won',
 'pass_angle',
 'pass_assisted_shot_id',
 'pass_body_part',
 'pass_cross',
 'pass_cut_back',
 'pass_deflected',
 'pass_duel_win_probability',
 'pass_end_location',
 'pass_height',
 'pass_inswinging',
 'pass_length',
 'pass_no_touch',
 'pass_outcome',
 'pass_outswinging',
 'pass_pass_cluster_id',
 'pass_pass_cluster_label',
 'pass_pass_cluster_probability',
 'pass_pass_success_probability',
 'pass_recipient',
 'pass_recipient_id',
 'pass_shot_assist',
 'pass_switch',
 'pass_technique',
 'pass_through_ball',
 'pass_type',
 'pass_xclaim',
 'shot_key_pass_id',
 'pass_goal_assist',
 'pass_straight',
 'pass_miscommunication']

In [17]:
# the core columns that every event has
core_columns = ["match_id", "period", "minute", "second", "team", "player",
                "type", "location", "possession_team", "play_pattern"]
events[core_columns].head(10)

,match_id,period,minute,second,team,player,type,location,possession_team,play_pattern
0,3981145,1,0,0,Nashville SC,NaN,Starting XI,NaN,Nashville SC,Other
1,3981145,1,0,0,New England Revolution,NaN,Starting XI,NaN,Nashville SC,Other
2,3981145,1,0,0,Nashville SC,NaN,Half Start,NaN,Nashville SC,Other
3,3981145,1,0,0,New England Revolution,NaN,Half Start,NaN,Nashville SC,Other
4,3981145,2,45,0,Nashville SC,NaN,Half Start,NaN,New England Revolution,Other
5,3981145,2,45,0,New England Revolution,NaN,Half Start,NaN,New England Revolution,Other
6,3981145,1,0,0,New England Revolution,Leonardo Campana Romero,Pass,"[60.0, 40.0]",New England Revolution,From Kick Off
7,3981145,1,0,3,New England Revolution,Jackson William Yueill,Pass,"[46.5, 43.1]",New England Revolution,From Kick Off
8,3981145,1,0,6,New England Revolution,Brandon Scott Bye,Pass,"[39.3, 62.7]",New England Revolution,From Kick Off
9,3981145,1,0,8,New England Revolution,Mamadou Fofana,Pass,"[27.0, 46.8]",New England Revolution,From Kick Off


In [18]:
events["type"].value_counts()

type
Pass                 50788
Ball Receipt*        46503
Carry                41019
Pressure             17938
Ball Recovery         4305
Duel                  2576
Block                 1942
Clearance             1519
Goal Keeper           1486
Foul Committed        1333
Foul Won              1265
Dribble               1229
Shot                  1166
Miscontrol            1104
Dispossessed          1015
Interception           786
Dribbled Past          694
Substitution           408
50/50                  318
Half Start             192
Half End               192
Injury Stoppage        146
Tactical Shift         136
Starting XI             96
Referee Ball-Drop       79
Shield                  56
Error                   46
Bad Behaviour           43
Player Off              33
Player On               31
Offside                 14
Own Goal Against         4
Own Goal For             4
Name: count, dtype: int64

### Location text to x, y
Locations are stored as text like "[61.0, 40.1]". Pitch is 120 × 80, every team attacks left → right.

In [19]:
example = events["location"].dropna().iloc[0]
print(example, "| type:", type(example))

[60.0, 40.0] | type: <class 'str'>


In [20]:
location_text = events["location"].str.strip("[]")          # "61.0, 40.1"
location_parts = location_text.str.split(",", expand=True)  # two columns: "61.0" and " 40.1"

events["x"] = location_parts[0].astype(float)
events["y"] = location_parts[1].astype(float)

events[["type", "location", "x", "y"]].dropna().head()

,type,location,x,y
6,Pass,"[60.0, 40.0]",60.0,40.0
7,Pass,"[46.5, 43.1]",46.5,43.1
8,Pass,"[39.3, 62.7]",39.3,62.7
9,Pass,"[27.0, 46.8]",27.0,46.8
10,Pass,"[28.1, 30.0]",28.1,30.0


### End location to end_x, end_y
Passes and carries store their end point in different columns. Combine them into one, then split into numbers.
Used later for final-third entries, corner targets and goal kicks.

In [21]:
end_location_text = events["pass_end_location"].fillna(events["carry_end_location"])  # pass end, or carry end if not a pass
end_location_text = end_location_text.str.strip("[]")
end_location_parts = end_location_text.str.split(",", expand=True)

events["end_x"] = end_location_parts[0].astype(float)
events["end_y"] = end_location_parts[1].astype(float)

### Add match info to events

In [22]:
match_info = matches[["match_id", "match_date", "home_team", "away_team", "focus_team"]]
events = events.merge(match_info, on="match_id", how="left")

events["is_home"] = events["team"] == events["home_team"]
events["opponent"] = np.where(events["is_home"], events["away_team"], events["home_team"])

### Focus team vs team
- Focus team: whose match
- Team: who made the action

San Diego's own actions show how they attack. Opponents' actions show how teams hurt them.

In [23]:
sd_match_events = events[events["focus_team"] == "San Diego"]

is_sd_action = sd_match_events["team"] == "San Diego FC"
sd_own_events = sd_match_events[is_sd_action]
opponent_events_vs_sd = sd_match_events[~is_sd_action]

print("All events in SD matches:", len(sd_match_events))
print("SD's own events:         ", len(sd_own_events))
print("Opponent events vs SD:   ", len(opponent_events_vs_sd))

All events in SD matches: 90782
SD's own events:          51884
Opponent events vs SD:    38898


In [24]:
sd_match_events["team"].value_counts()

team
San Diego FC            51884
Real Salt Lake           3479
Seattle Sounders         3456
LA Galaxy                3436
FC Dallas                3265
Vancouver Whitecaps      3230
Colorado Rapids          3071
St. Louis City           2983
Austin FC                2716
Charlotte                1923
Columbus Crew            1919
LAFC                     1853
Houston Dynamo           1698
Chicago Fire             1671
Minnesota United         1424
Sporting Kansas City     1407
Toronto FC               1367
Name: count, dtype: int64

## 4 - San Diego results

In [25]:
sd_results = matches[matches["focus_team"] == "San Diego"].copy()
sd_results["sd_home"] = sd_results["home_team"] == "San Diego FC"

In [26]:
# goals for / against depend on whether SD was home or away
sd_results["opponent"] = np.where(sd_results["sd_home"], sd_results["away_team"], sd_results["home_team"])
sd_results["goals_for"] = np.where(sd_results["sd_home"], sd_results["home_score"], sd_results["away_score"])
sd_results["goals_against"] = np.where(sd_results["sd_home"], sd_results["away_score"], sd_results["home_score"])

In [27]:
def get_result(row):
    if row["goals_for"] > row["goals_against"]:
        return "W"
    elif row["goals_for"] == row["goals_against"]:
        return "D"
    else:
        return "L"

sd_results["result"] = sd_results.apply(get_result, axis=1)   # axis=1: run the function on each row
sd_results["points"] = sd_results["result"].map({"W": 3, "D": 1, "L": 0})

sd_results[["match_date", "opponent", "sd_home", "goals_for", "goals_against", "result"]]

,match_date,opponent,sd_home,goals_for,goals_against,result
1,2025-02-24,LA Galaxy,False,2.0,0.0,W
2,2025-03-02,St. Louis City,True,0.0,0.0,D
5,2025-03-09,Real Salt Lake,False,3.0,1.0,W
6,2025-03-16,Columbus Crew,True,1.0,1.0,D
9,2025-03-23,Austin FC,False,1.0,2.0,L
11,2025-03-30,LAFC,True,3.0,2.0,W
13,2025-04-06,Seattle Sounders,True,3.0,0.0,W
14,2025-04-13,Colorado Rapids,False,2.0,3.0,L
17,2025-04-19,Charlotte,False,0.0,3.0,L
18,2025-04-26,Real Salt Lake,True,1.0,3.0,L


Home vs away record

In [28]:
home_away_record = sd_results.groupby("sd_home")[["points", "goals_for", "goals_against"]].mean()
home_away_record.round(2)

,points,goals_for,goals_against
sd_home,,,
False,1.91,2.18,1.73
True,1.69,1.77,1.00


In [29]:
sd_results["result"].value_counts()

result
W    13
L     7
D     4
Name: count, dtype: int64

### Goals vs xG

In [30]:
shots_in_sd_matches = sd_match_events[sd_match_events["type"] == "Shot"]

sd_shots = shots_in_sd_matches[shots_in_sd_matches["team"] == "San Diego FC"]
opponent_shots = shots_in_sd_matches[shots_in_sd_matches["team"] != "San Diego FC"]

In [31]:
number_of_matches = len(sd_results)

sd_xg_per_game = sd_shots["shot_statsbomb_xg"].sum() / number_of_matches
opponent_xg_per_game = opponent_shots["shot_statsbomb_xg"].sum() / number_of_matches
sd_goals_per_game = sd_results["goals_for"].mean()
opponent_goals_per_game = sd_results["goals_against"].mean()

print(f"SD goals per game:       {sd_goals_per_game:.2f}   |  SD xG per game:       {sd_xg_per_game:.2f}")
print(f"Conceded per game:       {opponent_goals_per_game:.2f}   |  xG against per game:  {opponent_xg_per_game:.2f}")

SD goals per game:       1.96   |  SD xG per game:       1.53
Conceded per game:       1.33   |  xG against per game:  1.33


San Diego score 1.96 goals per game from 1.53 xG, about 0.4 above. Goals conceded match xG against (1.33). Which players are above their xG: notebook 02.

## 5 - SkillCorner

In [32]:
physical_tables = []
for file in sorted(PHYSICAL_FOLDER.glob("*.csv")):
    physical_tables.append(pd.read_csv(file))

physical = pd.concat(physical_tables, ignore_index=True)
physical.shape

(1369, 93)

In [33]:
physical.info()

<class 'pandas.DataFrame'>
RangeIndex: 1369 entries, 0 to 1368
Data columns (total 93 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   player_name                                1369 non-null   str    
 1   player_short_name                          1369 non-null   str    
 2   player_id                                  1369 non-null   int64  
 3   player_birthdate                           1369 non-null   str    
 4   team_name                                  1369 non-null   str    
 5   team_id                                    1369 non-null   int64  
 6   match_name                                 1369 non-null   str    
 7   match_id                                   1369 non-null   int64  
 8   match_date                                 1369 non-null   str    
 9   competition_name                           1369 non-null   str    
 10  competition_id                     

In [34]:
physical_preview_columns = ["player_short_name", "team_name", "match_date", "position_group",
                            "minutes_full_all", "total_distance_full_all", "sprint_count_full_all",
                            "physical_check_passed"]
physical[physical_preview_columns].head()

,player_short_name,team_name,match_date,position_group,minutes_full_all,total_distance_full_all,sprint_count_full_all,physical_check_passed
0,A. Qasem,Nashville,2025-02-23,Wide Attacker,16.50,NaN,NaN,False
1,A. Muyl,Nashville,2025-02-23,Wide Attacker,76.55,NaN,NaN,False
2,A. Yusuf,New England Revolution,2025-02-23,Midfield,93.05,NaN,NaN,False
3,A. Najar,Nashville,2025-02-23,Full Back,93.05,NaN,NaN,False
4,B. Bye,New England Revolution,2025-02-23,Full Back,93.05,NaN,NaN,False


Some rows fail SkillCorner's quality check and have missing distances. Filtered out before using the physical data.

In [35]:
physical["physical_check_passed"].value_counts()

physical_check_passed
True     1338
False      31
Name: count, dtype: int64

In [36]:
dynamic_tables = []
for file in sorted(SKILLCORNER_FOLDER.glob("*.csv")):
    dynamic_tables.append(pd.read_csv(file, low_memory=False))

dynamic_events = pd.concat(dynamic_tables, ignore_index=True)
dynamic_events.shape

(204707, 269)

In [37]:
dynamic_events["event_type"].value_counts()

event_type
passing_option       131766
player_possession     47741
off_ball_run          25200
Name: count, dtype: int64

In [38]:
off_ball_runs = dynamic_events[dynamic_events["event_type"] == "off_ball_run"]
off_ball_runs["event_subtype"].value_counts()

event_subtype
run_ahead_of_the_ball    7381
support                  4310
coming_short             3189
dropping_off             2453
behind                   2349
cross_receiver           2196
pulling_wide             1486
pulling_half_space        684
overlap                   669
underlap                  483
Name: count, dtype: int64

## 6 - Save
Parquet files in data/processed

In [39]:
PROCESSED_FOLDER = DATA_FOLDER / "processed"
PROCESSED_FOLDER.mkdir(exist_ok=True)

matches.to_parquet(PROCESSED_FOLDER / "matches.parquet")
events.to_parquet(PROCESSED_FOLDER / "sb_events.parquet")
physical.to_parquet(PROCESSED_FOLDER / "physical.parquet")
dynamic_events.to_parquet(PROCESSED_FOLDER / "sc_dynamic.parquet")

print("Saved files:")
for file in sorted(PROCESSED_FOLDER.iterdir()):
    print(" ", file.name)

Saved files:
  asa_players_2025_h2.parquet
  asa_players_2026.parquet
  asa_team_possession_2026.parquet
  ex2_acquisition_table.csv
  ex2_coach_numbers.csv
  ex2_peer_group_scored.parquet
  ex2_pool.parquet
  ex2_pool_checked.parquet
  ex2_pool_for_transfermarkt.csv
  ex2_pool_summary.csv
  matches.parquet
  physical.parquet
  sb_events.parquet
  sc_dynamic.parquet
